# ChromaGraphNet quickstart

This notebook walks through:

1. Building a ChromaGraphNet model from the default config.
2. Generating synthetic inputs at the right shapes.
3. Constructing the chromatin graph with `build_graph_for_window`.
4. Running a forward pass and inspecting all five output heads.
5. Estimating epistemic uncertainty with MC dropout.
6. Loading the v0.1 random-init checkpoint for comparison.

All cells run on CPU in a few seconds.

In [ ]:
import torch
from chromagraphnet import (
    ChromaGraphNet,
    ChromaGraphNetConfig,
    build_graph_for_window,
    batch_graphs,
    load_model,
)

torch.manual_seed(0)

# A smaller config so the notebook runs fast on CPU. For real use,
# stick with the default 4.01 Mb / 401-bin config.
cfg = ChromaGraphNetConfig()
cfg.backbone.receptive_field_bp = 400_000
cfg.backbone.n_anchor_bins = 40
cfg.backbone.vstripe_length = 21
cfg.backbone.acc_num_conv1d_layers = 4
cfg.fusion.n_latents = 16
cfg.fusion.n_layers = 1
cfg.graph.n_layers = 2
cfg.heads.n_anchor_bins = 40
cfg.heads.vstripe_length = 21
cfg.fusion.n_anchor_bins = 40
cfg.modalities.n_anchor_bins = 40
cfg.__post_init__()

model = ChromaGraphNet(cfg)
model.eval()
print(f'Model parameters: {model.num_parameters():,}')

## Generate synthetic inputs

Replace these with real preprocessed data. See `docs/data_format.md` for the schema.

In [ ]:
B = 1
L = cfg.backbone.n_anchor_bins

acc_ctcf = torch.randn(B, 2, cfg.backbone.n_fine_bins)
coacc    = torch.randn(B, 40, cfg.backbone.n_coacc_bins)
rna      = torch.randn(B, L, 1)
chip     = torch.randn(B, 5, cfg.backbone.n_fine_bins)
motif    = torch.randn(B, L, 200)

print(f'acc_ctcf: {tuple(acc_ctcf.shape)}')
print(f'coacc:    {tuple(coacc.shape)}')
print(f'rna:      {tuple(rna.shape)}')
print(f'chip:     {tuple(chip.shape)}')
print(f'motif:    {tuple(motif.shape)}')

## Build the chromatin graph

The graph builder accepts three optional inputs (co-accessibility, Hi-C prior, CTCF orientation) and always adds the genomic-adjacency backbone.

In [ ]:
edge_index, edge_attr = build_graph_for_window(
    n_anchor_bins=L,
    coaccessibility=torch.rand(L, L),
    hic_prior=torch.rand(L, L) * 2.0,
    ctcf_orientation=torch.randint(-1, 2, (L,)),
    coacc_topk=5,
    hic_threshold=1.0,
)
print(f'edge_index: {tuple(edge_index.shape)}  ({edge_index.shape[1]} edges)')
print(f'edge_attr:  {tuple(edge_attr.shape)}')

## Run a forward pass

In [ ]:
with torch.no_grad():
    out = model(
        acc_ctcf=acc_ctcf, coacc=coacc, rna=rna, chip=chip, motif=motif,
        edge_index=edge_index, edge_attr=edge_attr,
    )

for k, v in out.items():
    print(f'{k:25s} {tuple(v.shape)}')

## Estimate uncertainty with MC dropout

In [ ]:
out_unc = model.predict(
    acc_ctcf=acc_ctcf, coacc=coacc, rna=rna, chip=chip, motif=motif,
    edge_index=edge_index, edge_attr=edge_attr,
    return_uncertainty=True, n_uncertainty_samples=20,
)

for k in ('contact_map_mean', 'contact_map_std',
          'compartment_logits_mean', 'compartment_logits_std'):
    v = out_unc[k]
    print(f'{k:32s} {tuple(v.shape)}  range=[{v.min():.3f}, {v.max():.3f}]')

## Visualize the predicted contact map

Run `pip install matplotlib` if not already installed.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(out['contact_map'][0].numpy(), cmap='RdBu_r')
    axes[0].set_title('Predicted contact map')
    axes[1].imshow(out_unc['contact_map_std'][0].numpy(), cmap='viridis')
    axes[1].set_title('MC-dropout std')
    for ax in axes:
        ax.set_xlabel('anchor bin')
        ax.set_ylabel('anchor bin')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed; skipping plot.')

## Next steps

- Replace the synthetic inputs with preprocessed real data following `docs/data_format.md`.
- Try different config overrides (e.g. `cfg.use_graph = False` for ChromaFold-only fallback).
- Wait for the v0.2 release for trained weights, or train your own with the (forthcoming) training pipeline.